# Zero-Shot & Few-Shot Defect Segmentation for Smartphone Screens
### DINOv2 memory bank + calibrated decisions + SAM 2 — phase-wise pipeline (v2)

Run in **Google Colab with a GPU** (Runtime → Change runtime type → T4 GPU). Run the phases in order;
each phase saves its results to `results/phaseN_*.json`, so you can stop and resume.

| Phase | What it does | Fixes |
|---|---|---|
| 0 | Setup, upload project | — |
| 1 | Reference bank, JPEG matching, **label-free calibration**, domain-shift diagnostic | always-"defect" bug, PNG/JPEG shift |
| 2 | Honest evaluation: raw-score AUROC/PRO, image-level AUROC, peaks vs regions | inflated / incomplete metrics |
| 3 | SAM 2 with box + positive/negative prompts; **mask metrics** (IoU/Dice) vs heatmap | unevaluated, unfiltered SAM 2 |
| 4 | Feature ablation on **val** (aspect vs square, registers, k), bank size | resolution loss, untuned settings |
| 5 | Live demo: screen ROI + few-shot bank of *your* phone | fingers/background, domain gap |
| 6 | Final summary tables for the report | — |

## Phase 0 — Setup

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"
!pip install -q "git+https://github.com/facebookresearch/sam2.git"
!mkdir -p checkpoints
!test -f checkpoints/sam2.1_hiera_small.pt || wget -q -O checkpoints/sam2.1_hiera_small.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
print("Setup complete.")

Get the project into Colab — pick **one** of the two cells. The zip must contain `src/` and `data/raw/MSD/`.

In [ ]:
# OPTION A: upload a zip of the project folder
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
!unzip -q -o "$zip_name" -d /content/project
%cd /content/project

In [ ]:
# OPTION B: Google Drive (edit the path)
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Computer Vision

In [ ]:
import os, sys, json, time
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.config import PipelineConfig, BASELINE_CONFIG
from src.data.msd_dataset import MSDDataset, DEFECT_CLASSES
from src.feature_extraction.dinov2_extractor import DinoV2Extractor
from src.inference import build_reference_inspector, build_few_shot_inspector
from src.calibration.calibration import score_distribution_diagnostic
from src.experiments import (build_eval_set, heatmaps_for, run_heatmap_eval, run_image_level_eval,
                             exp_A_roi, exp_B_thresholds, exp_C_peaks_vs_regions, exp_D_F_masks,
                             exp_E_bank_size, save_json)
from src.visualize import show_inspection, show_roi

os.makedirs('results/figures', exist_ok=True)
cfg = PipelineConfig()          # improved defaults; Phase 4 checks them on the val split
print(json.dumps(cfg.to_dict(), indent=1))
dataset = MSDDataset('data/raw/MSD')
split = dataset.split(seed=cfg.seed)
print({k: len(v) for k, v in split.items()}, '| normal reference images:', len(dataset.normal_samples()))

## Phase 1 — Reference bank, JPEG matching and label-free calibration

* MSD's 20 normal images are **PNG**, all defect images are **JPEG**. We estimate the defect images' JPEG
  quality from their quantization tables and re-encode the normals to match, so compression artifacts
  are not scored as anomalies.
* Each normal image is scored **leave-one-out** (against the other 19). Their scores define the thresholds:
  **image threshold** = the highest score ever seen on a clean screen; **pixel threshold** = 99.5th
  percentile of clean pixels. No defect labels are used.

In [ ]:
print('Estimated JPEG quality of defect images:', dataset.estimate_defect_jpeg_quality())
extractor = DinoV2Extractor.from_config(cfg)
t0 = time.time()
inspector = build_reference_inspector(extractor, dataset, cfg)
cal = inspector.calibration
print(f'Memory bank: {inspector.bank.size} x {inspector.bank.embed_dim} from {inspector.bank.n_images} images '
      f'({time.time()-t0:.1f}s)')
print(f'pixel threshold (low) = {cal.pixel_threshold:.4f} | image threshold (high) = {cal.image_threshold:.4f}')
print('LOO normal image scores:', np.round(cal.details['normal_image_scores'], 4))

In [ ]:
# Domain-shift diagnostic on the VAL split: do defect images' normal-looking areas score like clean screens?
es_val = build_eval_set(extractor, dataset, split['val'], cfg, 'val')
hm_val = heatmaps_for(es_val, inspector.bank, cfg)
diag = score_distribution_diagnostic(inspector.normal_loo_heatmaps, hm_val, es_val.gts, cal)
print('median background pixel sits at percentile', round(diag['background_median_percentile_in_normals'], 1),
      'of clean-screen scores (≈50 = no shift, ≈100 = strong shift)')
print('WARNING:', diag['warning']) if diag['warning'] else print('No domain-shift warning.')

plt.figure(figsize=(7, 3.5))
bins = np.linspace(0, np.percentile(np.concatenate([h.ravel() for h in hm_val]), 99.9), 120)
plt.hist(np.concatenate([h.ravel() for h in inspector.normal_loo_heatmaps]), bins, density=True, alpha=.6, label='clean screens (LOO)')
bg = np.concatenate([h[g == 0] for h, g in zip(hm_val, es_val.gts)])
fg = np.concatenate([h[g == 1] for h, g in zip(hm_val, es_val.gts)])
plt.hist(bg, bins, density=True, alpha=.5, label='defect images: background')
plt.hist(fg, bins, density=True, alpha=.5, label='defect pixels')
for t, n in ((cal.pixel_threshold, 'pixel thr'), (cal.image_threshold, 'image thr')):
    plt.axvline(t, ls='--', c='k'); plt.text(t, plt.ylim()[1]*.9, n, rotation=90)
plt.yscale('log'); plt.xlabel('raw cosine-distance score'); plt.legend(fontsize=7); plt.tight_layout()
plt.savefig('results/figures/phase1_score_distributions.png', dpi=150); plt.show()

save_json({'config': cfg.to_dict(), 'memory_bank': {'size': inspector.bank.size, 'dim': inspector.bank.embed_dim},
           'calibration': cal.to_dict(), 'diagnostic_val': diag}, 'results/phase1_calibration.json')

## Phase 2 — Honest evaluation of the heatmap stage

* AUROC/PRO on **raw** scores (comparable to PatchCore-style literature), shown next to the original
  per-image-normalized protocol so the difference is visible.
* **Image-level** normal-vs-defective AUROC — the question the old pipeline could not answer.
* **Experiment C**: legacy "peaks above 0.6 on a normalized map" vs calibrated regions, including how often
  a *clean* screen would be flagged.

In [ ]:
es_test = build_eval_set(extractor, dataset, split['test'], cfg, 'test')
hm_test = heatmaps_for(es_test, inspector.bank, cfg)
heat = run_heatmap_eval(es_test, hm_test)
print(f"{'set':>8} | raw AUROC | raw PRO | old-protocol AUROC | old-protocol PRO")
for k in ['all', *DEFECT_CLASSES]:
    r = heat[k]
    print(f"{k:>8} | {r['raw_pixel_auroc']:.4f}    | {r['raw_pro_score']:.4f}  | "
          f"{r['per_image_normalized_pixel_auroc']:.4f}             | {r['per_image_normalized_pro_score']:.4f}")
img_level = run_image_level_eval(inspector.bank, inspector.normal_loo_heatmaps, es_test, cfg, cal)
print('\nImage-level AUROC (20 clean vs', img_level['n_defect'], 'defective):', round(img_level['image_auroc'], 4))
print('At the calibrated threshold:', {k: round(v, 3) if isinstance(v, float) else v
                                       for k, v in img_level['at_calibrated_threshold'].items()})

In [ ]:
exp_c = exp_C_peaks_vs_regions(es_test, hm_test, inspector.normal_loo_heatmaps, cfg, cal)
print(f"{'method':>26} | prompts/img | on-defect | GT regions hit | CLEAN images flagged")
for r in exp_c['rows']:
    print(f"{r['method']:>26} | {r['prompts_per_image']:11.2f} | {r['prompt_precision']:9.3f} | "
          f"{r['gt_region_recall']:14.3f} | {r['clean_images_flagged']:.2f}")
save_json({'heatmap': heat, 'image_level': img_level, 'exp_C': exp_c}, 'results/phase2_evaluation.json')

## Phase 3 — SAM 2 with better prompts, evaluated with mask metrics

Each calibrated region gets a **box + spread positive points + negative points** on clearly-normal pixels.
SAM 2 returns 3 candidate masks; one is accepted only if it lies mostly **inside** the heatmap region
(rejects "whole screen" / "finger" masks), otherwise the heatmap region is kept (counted as a fallback).
Binary masks are scored with **IoU / Dice / precision / recall**, since AUROC/PRO are meaningless on 0/1 masks.

In [ ]:
from src.sam2_prompt.sam2_predictor import Sam2DefectSegmenter
segmenter = Sam2DefectSegmenter('checkpoints/sam2.1_hiera_small.pt', 'configs/sam2.1/sam2.1_hiera_s.yaml')
inspector.segmenter = segmenter

# Visual check: a few test images per class, plus two clean references (should say NORMAL)
picks = [s for c in DEFECT_CLASSES for s in [x for x in split['test'] if x.label == c][:2]]
fig, axes = plt.subplots(len(picks), 5, figsize=(21, 3.4 * len(picks)))
for row, s in enumerate(picks):
    res = inspector.inspect(dataset.load_image(s))
    gt = es_test.gts[es_test.samples.index(s)]
    show_inspection(res, gt=gt, title=f'{s.label}: {res.summary()[:9]}', axes=axes[row], show=False)
plt.tight_layout(); plt.savefig('results/figures/phase3_examples.png', dpi=110); plt.show()

In [ ]:
# Experiment B (VAL): choice of the pixel (low) threshold
exp_b = exp_B_thresholds(es_val, hm_val, inspector.normal_loo_heatmaps, cfg)
print(f"{'method':>46} | thr    | Dice  | IoU   | regions/img")
for r in exp_b['rows']:
    print(f"{r['method']:>46} | {r['pixel_threshold']:.4f} | {r['pooled_dice']:.3f} | {r['pooled_iou']:.3f} | {r['mean_regions_per_image']:.2f}")

In [ ]:
# Experiments D + F (TEST): legacy SAM 2 vs each prompt strategy vs thresholded heatmap.
# Runs with the label-free thresholds. Set LIMIT = 60 for a quick pass.
LIMIT = None
exp_df = exp_D_F_masks(es_test, hm_test, dataset, cfg, cal, segmenter, limit=LIMIT)
print(f"{'method':>24} | Dice  | IoU   | prec  | recall | GT-regions hit | FP comps/img | fallback")
for name, r in exp_df['methods'].items():
    a = r['all']
    print(f"{name:>24} | {a['pooled_dice']:.3f} | {a['pooled_iou']:.3f} | {a['pooled_precision']:.3f} | "
          f"{a['pooled_recall']:.3f}  | {a['gt_region_recall']:.3f}          | "
          f"{a['false_positive_components_per_image']:.2f}         | {r.get('sam_fallback_rate', float('nan')):.2f}")
print('\nPer class Dice:')
for name, r in exp_df['methods'].items():
    print(f"{name:>24}", ' '.join(f"{c}={r[c]['pooled_dice']:.3f}" for c in DEFECT_CLASSES if c in r))
save_json({'exp_B_val': exp_b, 'exp_DF_test': exp_df}, 'results/phase3_sam2.json')

## Phase 4 — Feature ablation on VAL, then bank size

Settings are chosen on **val** (not test) to keep the test numbers honest. The val subset keeps this to a
few minutes per configuration. Afterwards, if a different configuration wins, Phase 2 is re-run on test
with it.

In [ ]:
VAL_PER_CLASS = 30
val_sub = [s for c in DEFECT_CLASSES for s in [x for x in split['val'] if x.label == c][:VAL_PER_CLASS]]
ablation_cfgs = {
    'original (square 518, k=9, no JPEG match)': BASELINE_CONFIG,
    'aspect 1036, k=1, no JPEG match': cfg.with_(normal_jpeg_quality=None),
    'aspect 1036, k=1':             cfg,
    'aspect 1036, k=9':             cfg.with_(k=9),
    'aspect 1036, registers, k=1':  cfg.with_(model_name='dinov2_vits14_reg'),
    'aspect 1036, layers 8+11':     cfg.with_(layers=(8, 11)),
}
exp_g = {}
for name, c in ablation_cfgs.items():
    t0 = time.time()
    ext_c = DinoV2Extractor.from_config(c)
    insp_c = build_reference_inspector(ext_c, dataset, c)
    es_c = build_eval_set(ext_c, dataset, val_sub, c, 'val_sub', progress=False)
    r = run_heatmap_eval(es_c, heatmaps_for(es_c, insp_c.bank, c), include_normalized=False)
    exp_g[name] = {'config': c.to_dict(), **{k: r[k] for k in ['all', *DEFECT_CLASSES]}, 's': round(time.time()-t0, 1)}
    print(f"{name:>42} | AUROC {r['all']['raw_pixel_auroc']:.4f} | PRO {r['all']['raw_pro_score']:.4f} | "
          f"scratch PRO {r['scratch']['raw_pro_score']:.4f} | {exp_g[name]['s']}s")
    del ext_c, insp_c, es_c
best_name = max(exp_g, key=lambda n: exp_g[n]['all']['raw_pro_score'])
print('\nBest on val (by PRO):', best_name)
save_json(exp_g, 'results/phase4_feature_ablation_val.json')

In [ ]:
# If a different configuration won on val, re-run the TEST heatmap evaluation with it (Phase 2 numbers).
best_cfg = ablation_cfgs[best_name]
if best_cfg.to_dict() != cfg.to_dict():
    cfg = best_cfg
    extractor = DinoV2Extractor.from_config(cfg)
    inspector = build_reference_inspector(extractor, dataset, cfg, segmenter)
    cal = inspector.calibration
    es_test = build_eval_set(extractor, dataset, split['test'], cfg, 'test')
    hm_test = heatmaps_for(es_test, inspector.bank, cfg)
    heat = run_heatmap_eval(es_test, hm_test)
    img_level = run_image_level_eval(inspector.bank, inspector.normal_loo_heatmaps, es_test, cfg, cal)
    save_json({'config': cfg.to_dict(), 'heatmap': heat, 'image_level': img_level}, 'results/phase4_test_best_config.json')
    print('Re-evaluated test with', best_name, '-> AUROC', round(heat['all']['raw_pixel_auroc'], 4),
          'PRO', round(heat['all']['raw_pro_score'], 4), '| re-run the Phase 3 D+F cell for final mask numbers')
else:
    print('Default configuration was best on val; Phase 2/3 test numbers stand.')

In [ ]:
# Experiment E: how many clean reference images are needed? For n < 20 the unused normals are
# genuinely held-out clean screens, giving an honest false-alarm rate.
exp_e = exp_E_bank_size(es_test, inspector.bank, inspector.normal_grids, inspector.normal_sizes, cfg,
                        sizes=(3, 5, 10, 15, 20), repeats=3)
import collections
agg = collections.defaultdict(list)
for r in exp_e['rows']:
    agg[r['n_reference']].append(r)
print(" n  | AUROC  | PRO    | defects detected | held-out clean flagged | image AUROC (held-out)")
for n, rs in sorted(agg.items()):
    m = lambda k: np.nanmean([r.get(k, np.nan) for r in rs])
    print(f"{n:3d} | {m('raw_pixel_auroc'):.4f} | {m('raw_pro_score'):.4f} | {m('defect_images_detected'):.3f}"
          f"            | {m('heldout_normals_flagged'):.3f}                  | {m('image_auroc_heldout'):.4f}")
save_json(exp_e, 'results/phase4_bank_size.json')

## Phase 5 — Live demo on a real phone (ROI + few-shot)

The MSD memory bank describes MSD's industrial camera setup, not your phone and room. The fix is the
**few-shot** mode:

1. Take **5–10 photos of the clean demo phone** exactly as you will photograph it in the demo (same phone,
   lighting, distance, screen content; hold it naturally — fingers are handled by the ROI).
2. The pipeline finds the screen in each photo (SAM 2, prompted at the image centre — keep the screen
   centred), warps it flat, excludes fingers/bezel, builds a bank and calibrates from those photos.
3. Then photograph the phone with a real smudge / stain / scratch.

Phone-camera photos uploaded from your gallery generally work better than a laptop webcam.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import io

def capture_photo(quality=0.92):
    '''Opens the webcam with a Capture button; returns a PIL image.'''
    display(Javascript('''
      async function takePhoto(quality) {
        const div = document.createElement('div');
        const btn = document.createElement('button'); btn.textContent = 'Capture';
        const video = document.createElement('video'); video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia(
            {video: {width: {ideal: 1920}, height: {ideal: 1080}}});
        document.body.appendChild(div); div.appendChild(video); div.appendChild(btn);
        video.srcObject = stream; await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => btn.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth; canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop(); div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      }'''))
    data = eval_js(f'takePhoto({quality})')
    return Image.open(io.BytesIO(b64decode(data.split(',')[1]))).convert('RGB')

def upload_photos():
    '''Upload one or more photos from your computer/phone gallery.'''
    from google.colab import files
    up = files.upload()
    return [Image.open(io.BytesIO(b)).convert('RGB') for b in up.values()]

In [ ]:
# Step 1: clean reference photos of the demo phone. Use ONE of these two lines:
clean_photos = upload_photos()
# clean_photos = [capture_photo() for _ in range(6)]
print(len(clean_photos), 'clean photos')

live_cfg = cfg.with_(border_margin_frac=0.03)
live_inspector = build_few_shot_inspector(extractor, clean_photos, live_cfg, segmenter, roi_method='auto')
print('Few-shot bank:', live_inspector.bank.size, 'patches | thresholds:',
      round(live_inspector.calibration.pixel_threshold, 4), round(live_inspector.calibration.image_threshold, 4))
for photo, roi in zip(clean_photos, live_inspector.reference_rois):
    show_roi(photo, roi); plt.show()   # check: cyan outline on the screen, fingers dark in the valid mask

In [ ]:
# Step 2: test photos (clean AND defective). Give labels so false alarms can be counted.
test_photos = upload_photos()
labels = input(f'Labels for the {len(test_photos)} photos, comma separated (clean/defect): ').split(',')
labels = [l.strip() for l in labels]

for i, (p, l) in enumerate(zip(test_photos, labels)):
    res = live_inspector.inspect(p)
    show_inspection(res, title=f'#{i} [{l}] ' + ('DEFECTIVE' if res.is_defective else 'NORMAL'))
    plt.savefig(f'results/figures/phase5_live_{i}.png', dpi=110); plt.show()
    print(res.summary(), '|', res.info.get('warning', ''))

In [ ]:
# Experiment A: full image vs manual crop vs automatic ROI, all with the few-shot bank.
# Optional manual boxes (x0, y0, x1, y1) per photo in original-pixel coordinates; None to skip.
manual_boxes = None   # e.g. [(60, 100, 340, 560), None, ...]
exp_a = exp_A_roi(live_inspector, test_photos, labels, manual_boxes,
                  methods=('full', 'manual', 'auto') if manual_boxes else ('full', 'auto'))
for m, s in exp_a['summary'].items():
    print(f"{m:>6}: {s}")

# Issue 3 in numbers: the MSD bank vs the few-shot bank on the same live photos (both with auto ROI)
inspector.roi_method = 'auto'
msd_on_live = [inspector.inspect(p, roi_kwargs={'margin_frac': 0.03}) for p in test_photos]
inspector.roi_method = None
fs_flags = [r['is_defective'] for r in exp_a['rows'] if r['method'] == 'auto']
compare = [{'photo': i, 'label': l, 'msd_bank_flagged': bool(m.is_defective), 'few_shot_flagged': bool(f)}
           for i, (l, m, f) in enumerate(zip(labels, msd_on_live, fs_flags))]
for c in compare: print(c)
save_json({'exp_A': {k: v for k, v in exp_a.items() if k != '_results'}, 'msd_vs_few_shot': compare,
           'few_shot_calibration': live_inspector.calibration.to_dict()}, 'results/phase5_live.json')

## Phase 6 — Final summary for the report

In [ ]:
import glob
summary = {os.path.basename(f): json.load(open(f)) for f in sorted(glob.glob('results/phase*.json'))}
p2 = summary.get('phase4_test_best_config.json') or summary['phase2_evaluation.json']
print('FINAL CONFIG:', json.dumps(p2.get('config', cfg.to_dict())))
print('\nHeatmap stage (test, raw scores):')
for k in ['all', *DEFECT_CLASSES]:
    r = p2['heatmap'][k]
    print(f"  {k:>8}: pixel AUROC {r['raw_pixel_auroc']:.4f} | PRO {r['raw_pro_score']:.4f}")
il = p2['image_level']
print(f"Image level: AUROC {il['image_auroc']:.4f}; detected {il['at_calibrated_threshold']['tpr_defect_detected']:.3f} "
      f"of defective, flagged {il['at_calibrated_threshold']['fpr_normal_flagged']:.3f} of clean (n={il['n_normal']})")
if 'phase3_sam2.json' in summary:
    print('\nMasks (test):')
    for name, r in summary['phase3_sam2.json']['exp_DF_test']['methods'].items():
        print(f"  {name:>24}: Dice {r['all']['pooled_dice']:.3f} | IoU {r['all']['pooled_iou']:.3f}")
save_json(summary, 'results/final_summary.json')
!cd results && zip -q -r ../results_bundle.zip . && cd .. && echo "results_bundle.zip ready"

In [ ]:
from google.colab import files
files.download('results_bundle.zip')   # send this zip back to continue with the report